# Visualizing GIS Layers

This notebook demonstrates interactive visualization of geospatial data using Folium (Leaflet.js wrapper).

**What you'll learn:**
- Create interactive maps with Folium
- Add multiple layers with different styles
- Configure popups and tooltips
- Handle coordinate reference system transformations

In [ ]:
# Import required libraries
import geopandas as gpd
import folium
from folium import plugins
import warnings
warnings.filterwarnings('ignore')

## 1. Configure Data Source

In [ ]:
# Path to geodatabase
GDB_PATH = "../data-extracted/RVO_NW_I_GGM.gdb"

# Layers to visualize (modify these based on your data)
LAYERS_TO_SHOW = [
    # Add layer names from your geodatabase here
    # Example: "Wind_Farm_Zone", "Cables", "Boreholes"
]

## 2. Helper Functions

In [ ]:
def load_layer(gdb_path, layer_name):
    """
    Load a layer from geodatabase and transform to WGS84.
    
    Args:
        gdb_path: Path to the geodatabase
        layer_name: Name of the layer to load
        
    Returns:
        GeoDataFrame in EPSG:4326 (WGS84) or None if loading fails
    """
    try:
        gdf = gpd.read_file(gdb_path, layer=layer_name)
        
        # Filter invalid/empty geometries
        gdf = gdf[gdf.geometry.is_valid]
        gdf = gdf[~gdf.geometry.is_empty]
        
        # Transform to WGS84 for web mapping
        if gdf.crs and gdf.crs != "EPSG:4326":
            gdf = gdf.to_crs("EPSG:4326")
        
        print(f"Loaded {layer_name}: {len(gdf)} features")
        return gdf
    except Exception as e:
        print(f"Failed to load {layer_name}: {e}")
        return None

In [ ]:
def get_center_from_gdf(gdf):
    """
    Calculate the center point of a GeoDataFrame.
    
    Returns:
        Tuple of (latitude, longitude)
    """
    bounds = gdf.total_bounds  # [minx, miny, maxx, maxy]
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2
    return center_lat, center_lon

In [ ]:
def create_popup(row, fields=None):
    """
    Create HTML popup content for a feature.
    
    Args:
        row: A row from a GeoDataFrame
        fields: List of field names to include (None = all)
        
    Returns:
        HTML string for popup
    """
    if fields is None:
        fields = [c for c in row.index if c != 'geometry']
    
    html = "<div style='max-width: 300px;'>"
    for field in fields[:10]:  # Limit to first 10 fields
        if field in row.index:
            value = row[field]
            if value is not None and str(value) != 'nan':
                html += f"<b>{field}:</b> {value}<br>"
    html += "</div>"
    return html

## 3. Load Data

In [ ]:
import fiona

# List available layers
available_layers = fiona.listlayers(GDB_PATH)
print(f"Available layers: {len(available_layers)}")

# If no layers specified, try to load first few vector layers
if not LAYERS_TO_SHOW:
    # Try to find some layers to show
    for layer_name in available_layers[:5]:
        try:
            with fiona.open(GDB_PATH, layer=layer_name) as src:
                if src.schema.get('geometry') and len(src) > 0:
                    LAYERS_TO_SHOW.append(layer_name)
                    if len(LAYERS_TO_SHOW) >= 3:
                        break
        except:
            pass

print(f"\nLayers to visualize: {LAYERS_TO_SHOW}")

In [ ]:
# Load selected layers
loaded_layers = {}

for layer_name in LAYERS_TO_SHOW:
    gdf = load_layer(GDB_PATH, layer_name)
    if gdf is not None and len(gdf) > 0:
        loaded_layers[layer_name] = gdf

print(f"\nSuccessfully loaded {len(loaded_layers)} layers")

## 4. Create Basic Map

In [ ]:
# Determine map center from loaded data
if loaded_layers:
    first_layer = list(loaded_layers.values())[0]
    center_lat, center_lon = get_center_from_gdf(first_layer)
else:
    # Default center (North Sea)
    center_lat, center_lon = 52.5, 4.0

print(f"Map center: {center_lat:.4f}, {center_lon:.4f}")

In [ ]:
# Create base map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles='CartoDB positron'
)

# Add alternative tile layers
folium.TileLayer('OpenStreetMap').add_to(m)
folium.TileLayer('CartoDB dark_matter').add_to(m)

m

## 5. Add Layers with Styling

In [ ]:
# Color palette for different layers
COLORS = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf']

def get_style_function(color, fill_opacity=0.4):
    """Return a style function for polygons/lines."""
    return lambda x: {
        'fillColor': color,
        'color': color,
        'weight': 2,
        'fillOpacity': fill_opacity
    }

In [ ]:
# Create a new map with all layers
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles='CartoDB positron'
)

# Add each layer
for idx, (layer_name, gdf) in enumerate(loaded_layers.items()):
    color = COLORS[idx % len(COLORS)]
    geom_type = gdf.geometry.geom_type.iloc[0] if len(gdf) > 0 else None
    
    # Create a feature group for this layer
    fg = folium.FeatureGroup(name=layer_name)
    
    if geom_type in ['Point', 'MultiPoint']:
        # Add point markers
        for _, row in gdf.iterrows():
            if row.geometry:
                folium.CircleMarker(
                    location=[row.geometry.y, row.geometry.x],
                    radius=6,
                    color=color,
                    fill=True,
                    fillColor=color,
                    fillOpacity=0.7,
                    popup=folium.Popup(create_popup(row), max_width=300)
                ).add_to(fg)
    else:
        # Add polygon/line features
        folium.GeoJson(
            gdf.__geo_interface__,
            name=layer_name,
            style_function=get_style_function(color),
            tooltip=folium.GeoJsonTooltip(fields=gdf.columns[:3].tolist())
        ).add_to(fg)
    
    fg.add_to(m)
    print(f"Added layer: {layer_name} ({geom_type}, {len(gdf)} features)")

# Add layer control
folium.LayerControl().add_to(m)

m

## 6. Advanced Styling Examples

In [ ]:
# Example: Create a choropleth-style map if you have numeric data
# This shows how to style features based on attribute values

def create_choropleth_style(gdf, value_column, color_map='YlOrRd'):
    """
    Create a style function based on a numeric column.
    
    Args:
        gdf: GeoDataFrame
        value_column: Name of numeric column to use for coloring
        color_map: Matplotlib colormap name
        
    Returns:
        Style function for folium.GeoJson
    """
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    
    values = gdf[value_column].dropna()
    if len(values) == 0:
        return lambda x: {'fillColor': '#gray', 'color': 'gray'}
    
    vmin, vmax = values.min(), values.max()
    cmap = plt.cm.get_cmap(color_map)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    
    def style_function(feature):
        val = feature['properties'].get(value_column)
        if val is None:
            color = 'gray'
        else:
            color = mcolors.to_hex(cmap(norm(val)))
        return {
            'fillColor': color,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.7
        }
    
    return style_function

## 7. Add Map Plugins

In [ ]:
# Create a map with additional plugins
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles='CartoDB positron'
)

# Add minimap
minimap = plugins.MiniMap(toggle_display=True)
m.add_child(minimap)

# Add fullscreen button
plugins.Fullscreen().add_to(m)

# Add mouse position
plugins.MousePosition().add_to(m)

# Add measure control
plugins.MeasureControl(position='topleft').add_to(m)

# Add draw control for interactive drawing
plugins.Draw(export=True).add_to(m)

# Re-add layers
for idx, (layer_name, gdf) in enumerate(loaded_layers.items()):
    color = COLORS[idx % len(COLORS)]
    fg = folium.FeatureGroup(name=layer_name)
    
    geom_type = gdf.geometry.geom_type.iloc[0] if len(gdf) > 0 else None
    
    if geom_type in ['Point', 'MultiPoint']:
        for _, row in gdf.iterrows():
            if row.geometry:
                folium.CircleMarker(
                    location=[row.geometry.y, row.geometry.x],
                    radius=6,
                    color=color,
                    fill=True,
                    fillColor=color,
                    fillOpacity=0.7,
                    popup=folium.Popup(create_popup(row), max_width=300)
                ).add_to(fg)
    else:
        folium.GeoJson(
            gdf.__geo_interface__,
            name=layer_name,
            style_function=get_style_function(color)
        ).add_to(fg)
    
    fg.add_to(m)

folium.LayerControl().add_to(m)

m

## 8. Export Map

In [ ]:
# Save map to HTML file
output_file = 'interactive_map.html'
m.save(output_file)
print(f"Map saved to: {output_file}")

## Next Steps

- **Customize styles** - Modify colors, opacity, and line weights
- **Add more layers** - Update `LAYERS_TO_SHOW` with your layer names
- **Create choropleth maps** - Color features by attribute values
- **See spatial analysis** - Check `03_spatial_analysis.ipynb`